# Linear programming with the simplex method

`Simplex` is a two-phase primal simplex on a dense tableau. Phase 1 finds a
basic feasible solution by minimizing artificial variables; phase 2 optimizes
from there. Pivots follow Bland's rule, which guarantees termination even on
degenerate problems.

In [1]:
import numpy as np
from scipy.optimize import linprog

from mopt.linear import LPProblem, Simplex

## Standard form

`LPProblem` states

$$\min_x\; c^T x \quad\text{s.t.}\quad A_{ub} x \le b_{ub},\;\;
A_{eq} x = b_{eq},\;\; x \ge 0.$$

A maximization becomes a minimization by negating $c$.

In [2]:
# maximize 3x + 5y  s.t.  x <= 4,  2y <= 12,  3x + 2y <= 18
problem = LPProblem(c=[-3, -5], A_ub=[[1, 0], [0, 2], [3, 2]], b_ub=[4, 12, 18])
result = Simplex().solve(problem)

print(f"success = {result.success}   ({result.message})")
print(f"x       = {result.x}          (optimum is (2, 6))")
print(f"max     = {-result.fun}       (optimum is 36)")
print(f"pivots  = {result.n_iter}")

success = True   (Optimal solution found.)
x       = [2. 6.]          (optimum is (2, 6))
max     = 36.0       (optimum is 36)
pivots  = 3


## Equality constraints and negative right-hand sides

Both are handled by the standard-form conversion — equality rows go in
directly, and rows with a negative right-hand side are negated so phase 1 can
start.

In [3]:
# min 2x + 3y  s.t.  x + y = 10,  x <= 6   ->   (6, 4)
eq = Simplex().solve(LPProblem(c=[2, 3], A_ub=[[1, 0]], b_ub=[6],
                               A_eq=[[1, 1]], b_eq=[10]))
print(f"equality      x = {eq.x}, f = {eq.fun}")

# min x  s.t.  x >= 2, written as -x <= -2
neg = Simplex().solve(LPProblem(c=[1], A_ub=[[-1]], b_ub=[-2]))
print(f"negative rhs  x = {neg.x}, f = {neg.fun}")

equality      x = [6. 4.], f = 24.0
negative rhs  x = [2.], f = 2.0


## Infeasible and unbounded problems are reported

Neither raises — both come back as a result with `success=False` and a reason,
matching every other solver in the package.

In [4]:
infeasible = Simplex().solve(LPProblem(c=[1], A_ub=[[1], [-1]], b_ub=[1, -2]))
print(f"x <= 1 and x >= 2   -> success={infeasible.success}, {infeasible.message}")

unbounded = Simplex().solve(LPProblem(c=[-1, 0], A_ub=[[0, 1]], b_ub=[1]))
print(f"min -x, x unbounded -> success={unbounded.success}, {unbounded.message}")

x <= 1 and x >= 2   -> success=False, Problem is infeasible.
min -x, x unbounded -> success=False, Problem is unbounded.


## Degeneracy and cycling

At a degenerate vertex more constraints are active than the dimension
requires, and a naive pivot rule can cycle forever. Beale's example is the
classic case; Bland's rule brings it home.

In [5]:
degenerate = Simplex().solve(
    LPProblem(c=[-1, -1], A_ub=[[1, 0], [0, 1], [1, 1]], b_ub=[1, 1, 2]))
print(f"degenerate vertex -> f = {degenerate.fun}, pivots = {degenerate.n_iter}")

beale = LPProblem(
    c=[-0.75, 150, -0.02, 6],
    A_ub=[[0.25, -60, -0.04, 9], [0.5, -90, -0.02, 3], [0, 0, 1, 0]],
    b_ub=[0, 0, 1],
)
mine = Simplex().solve(beale)
ref = linprog(beale.c, A_ub=beale.A_ub, b_ub=beale.b_ub)
print(f"Beale's cycling example -> f = {mine.fun:.9f}  (scipy: {ref.fun:.9f})")

degenerate vertex -> f = -2.0, pivots = 3
Beale's cycling example -> f = -0.050000000  (scipy: -0.050000000)


## A random battery against SciPy

In [6]:
rng = np.random.default_rng(0)
worst = 0.0
for _ in range(50):
    n = int(rng.integers(2, 7))
    m = int(rng.integers(1, 9))
    # b >= 0 keeps x = 0 feasible; the x <= 10 rows keep the LP bounded
    p = LPProblem(
        c=rng.normal(size=n),
        A_ub=np.vstack([rng.normal(size=(m, n)), np.eye(n)]),
        b_ub=np.concatenate([rng.uniform(0.5, 5.0, size=m), np.full(n, 10.0)]),
    )
    mine = Simplex().solve(p)
    ref = linprog(p.c, A_ub=p.A_ub, b_ub=p.b_ub)
    assert mine.success and ref.success
    worst = max(worst, abs(mine.fun - ref.fun))
    assert (p.A_ub @ mine.x <= p.b_ub + 1e-8).all() and (mine.x >= -1e-8).all()

print(f"50 random LPs: all feasible and optimal, worst |f - f_scipy| = {worst:.2e}")

50 random LPs: all feasible and optimal, worst |f - f_scipy| = 3.20e-14
